# End-to-End Risk & Impact Prediction System

This notebook builds a reproducible pipeline to:
- Predict `risk_tier_3m` (ordinal class)
- Predict `risk_score_3m` (probability)
- Forecast `jobs_created_3m`, `jobs_lost_3m`, `revenue_3m`

It follows a strict time-based train/test protocol and avoids leakage.

## 1) Imports & Setup

In [1]:
import os
import json
import warnings
import subprocess
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, LinearRegression, PoissonRegressor, LassoCV
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    roc_auc_score,
    cohen_kappa_score,
    brier_score_loss,
    mean_squared_error,
    mean_absolute_error,
    classification_report,
)
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.inspection import PartialDependenceDisplay

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

try:
    import lightgbm as lgb
except Exception:
    lgb = None

try:
    import shap
except Exception:
    shap = None

try:
    from xgboost import XGBClassifier, XGBRegressor
except Exception:
    XGBClassifier, XGBRegressor = None, None


def detect_nvidia_gpu() -> bool:
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
            capture_output=True,
            text=True,
            check=False,
        )
        return result.returncode == 0 and bool(result.stdout.strip())
    except Exception:
        return False


def can_use_xgb_gpu() -> bool:
    return detect_nvidia_gpu() and XGBClassifier is not None and XGBRegressor is not None


def can_use_lgb_gpu() -> bool:
    return detect_nvidia_gpu() and lgb is not None


GPU_AVAILABLE = detect_nvidia_gpu()
USE_XGB_GPU = can_use_xgb_gpu()
USE_LGB_GPU = can_use_lgb_gpu()

BASE_DIR = Path("./ml")
DATA_DIR = BASE_DIR / "synthetic_outputs"
ARTIFACTS_DIR = BASE_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

CORE_PATH = DATA_DIR / "core_banking_loans.csv"
IMPACT_PATH = DATA_DIR / "impact_data.csv"

print(f"Core path exists: {CORE_PATH.exists()} -> {CORE_PATH}")
print(f"Impact path exists: {IMPACT_PATH.exists()} -> {IMPACT_PATH}")
print(f"Artifacts dir: {ARTIFACTS_DIR.resolve()}")
print(f"Data dir: {DATA_DIR.resolve()}")
print(f"GPU available: {GPU_AVAILABLE}")
print(f"XGBoost GPU enabled: {USE_XGB_GPU}")
print(f"LightGBM GPU enabled: {USE_LGB_GPU}")

Core path exists: True -> ml/synthetic_outputs/core_banking_loans.csv
Impact path exists: True -> ml/synthetic_outputs/impact_data.csv
Artifacts dir: /home/tr3p0l3m/Course Simulations/SEMESTER 4 SIMS/CAPSTONE/inkomoko-earlywarning-system/ml/artifacts
Data dir: /home/tr3p0l3m/Course Simulations/SEMESTER 4 SIMS/CAPSTONE/inkomoko-earlywarning-system/ml/synthetic_outputs
GPU available: True
XGBoost GPU enabled: True
LightGBM GPU enabled: True


## 2) Data Loading
Load files, inspect shape/head, and validate schema.

In [2]:
core_expected = [
    "loanNumber", "purpose", "strata", "clientId", "dateOfBirth", "cycle", "province", "district",
    "submissionDate", "approvalDate", "disbursementDate", "appliedAmount", "approvedAmount", "disbursedAmount",
    "loanType", "termsDuration", "actualPaymentAmount", "principalPaid", "interestPaid", "insuranceFeePaid",
    "totalLateFeesPaid", "excessAmountPaid", "interestWaived", "currentBalance", "principalBalance", "interestBalance",
    "feesBalance", "amountPastDue", "principalPastDue", "interestPastDue", "feesPastDue", "scheduledPrincipalAmount",
    "scheduledInterestAmount", "scheduledFeesAmount", "scheduledPaymentAmount", "lastPaymentAmount", "lastPrincipalAmount",
    "lastInterestAmount", "lastFeesAmount", "lastLateFeesAmount", "lastExcessAmount", "daysInArrears",
    "installmentInArrears", "lastPaymentDate", "loanStatus", "industrySectorOfActivity", "businessSubSector",
    "countrySpecific", "country_code"
]

impact_expected = [
    "unique_id", "survey_id", "survey_date", "age", "gender", "strata", "client_location", "nationality",
    "education_level", "business_sector", "business_sub_sector", "only_income_earner", "number_of_people_reponsible",
    "business_location", "is_business_registered", "has_access_to_finance_in_past_6months", "have_bank_account",
    "monthly_customer", "kept_sales_record", "bz_have_new_practices", "practice_type", "bz_have_new_product",
    "new_product_type", "job_created", "revenue", "hh_expense", "nps_detractor", "nps_passive", "nps_promoter",
    "satisfied_yes", "satisfied_no", "survey_name", "has_started_business", "why_not_started_business",
    "bz_source_capital", "did_you_buy_assets_in_the_past_6months", "business_challenges", "did_you_attended_all_training",
    "does_training_increased_bz_skills", "plan_after_program", "risk_tier_3m", "risk_score_3m", "jobs_created_3m",
    "jobs_lost_3m", "revenue_3m", "countrySpecific", "country_code"
]

core_df = pd.read_csv(CORE_PATH)
impact_df = pd.read_csv(IMPACT_PATH)

print("Core shape:", core_df.shape)
print("Impact shape:", impact_df.shape)

missing_core = sorted(set(core_expected) - set(core_df.columns))
missing_impact = sorted(set(impact_expected) - set(impact_df.columns))

print("Missing core columns:", missing_core)
print("Missing impact columns:", missing_impact)

display(core_df.head(3))
display(impact_df.head(3))

Core shape: (10000, 49)
Impact shape: (10000, 47)
Missing core columns: []
Missing impact columns: []


,loanNumber,purpose,strata,clientId,dateOfBirth,cycle,province,district,submissionDate,approvalDate,...,lastLateFeesAmount,lastExcessAmount,daysInArrears,installmentInArrears,lastPaymentDate,loanStatus,industrySectorOfActivity,businessSubSector,countrySpecific,country_code
0,LN-RW-2021-662506,Working capital,host,RW-RW-2023-S2-U02798,1972-05-19,6,Kigali City,Huye,2021-08-10,2021-09-06,...,0.00,1.23,0,0,2021-10-29,active,Retail & Trade,Grocery,Rwanda,RW
1,LN-ET-2026-513107,Emergency,refugee,ET-ET-2022-S2-U02282,1992-07-11,3,Tigray,Bole,2026-03-22,2026-03-30,...,0.57,8.88,3,3,2026-05-24,active,Personal Services,Cleaning,Ethiopia,ET
2,LN-KE-2026-493900,Working capital,host,KE-KE-2021-S3-U02632,1985-07-06,2,Nairobi,Kisumu,2026-08-20,2026-09-16,...,0.00,1.05,0,0,2027-01-16,active,Personal Services,Laundry,Kenya,KE


,unique_id,survey_id,survey_date,age,gender,strata,client_location,nationality,education_level,business_sector,...,did_you_attended_all_training,does_training_increased_bz_skills,plan_after_program,risk_tier_3m,risk_score_3m,jobs_created_3m,jobs_lost_3m,revenue_3m,countrySpecific,country_code
0,U02254,ET-2018-S3,2018-05-06,37,Male,refugee,Jijiga,Ethiopian,TVET,Agriculture,...,NaN,NaN,NaN,HIGH,0.761924,1,0,278.02,Ethiopia,ET
1,U02941,RW-2024-S1,2024-06-25,28,Female,refugee,Kigeme Camp,Rwandan,Diploma,ICT,...,NaN,NaN,NaN,MEDIUM,0.510883,0,0,372.06,Rwanda,RW
2,U00585,RW-2022-S3,2022-11-08,27,Female,host,Mahama Camp,Congolese,Primary,Hospitality and Tourism,...,True,True,Join cooperative,LOW,0.292304,2,0,876.59,Rwanda,RW


## 3) Data Cleaning
Handle missing values, type casting, and outlier clipping.

In [3]:
def to_datetime(df: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce")
    return df


def clip_iqr(series: pd.Series, whisker: float = 1.5) -> pd.Series:
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - whisker * iqr, q3 + whisker * iqr
    return series.clip(lower=low, upper=high)


core = core_df.copy()
impact = impact_df.copy()

core = to_datetime(core, ["submissionDate", "approvalDate", "disbursementDate", "lastPaymentDate", "dateOfBirth"])
impact = to_datetime(impact, ["survey_date"])

core["clientId"] = core["clientId"].astype(str).str.strip()
impact["unique_id"] = impact["unique_id"].astype(str).str.strip()

numeric_core_cols = [c for c in core.columns if c.lower().endswith("amount") or "balance" in c.lower() or "pastdue" in c.lower() or c in ["daysInArrears", "installmentInArrears", "cycle", "termsDuration"]]
numeric_impact_cols = ["age", "number_of_people_reponsible", "monthly_customer", "job_created", "revenue", "hh_expense", "risk_score_3m", "jobs_created_3m", "jobs_lost_3m", "revenue_3m"]

for col in numeric_core_cols:
    if col in core.columns:
        core[col] = pd.to_numeric(core[col], errors="coerce")
for col in numeric_impact_cols:
    if col in impact.columns:
        impact[col] = pd.to_numeric(impact[col], errors="coerce")

# Basic missing-value handling
for col in numeric_core_cols:
    if col in core.columns:
        core[col] = core[col].fillna(core[col].median())
for col in numeric_impact_cols:
    if col in impact.columns:
        impact[col] = impact[col].fillna(impact[col].median())

for col in ["strata", "loanType", "loanStatus", "industrySectorOfActivity", "businessSubSector", "country_code"]:
    if col in core.columns:
        core[col] = core[col].fillna("UNKNOWN").astype(str)
for col in ["strata", "gender", "education_level", "business_sector", "business_sub_sector", "country_code", "risk_tier_3m"]:
    if col in impact.columns:
        impact[col] = impact[col].fillna("UNKNOWN").astype(str)

# Outlier clipping on selected continuous fields
for col in ["revenue", "hh_expense", "monthly_customer", "risk_score_3m", "revenue_3m"]:
    if col in impact.columns:
        impact[col] = clip_iqr(impact[col])

print("Cleaning complete")
print("Core nulls:", int(core.isna().sum().sum()))
print("Impact nulls:", int(impact.isna().sum().sum()))

Cleaning complete
Core nulls: 0
Impact nulls: 86326


## 4) Feature Engineering
Repayment ratios, arrears trends, rolling averages, utilization, volatility, and sector benchmarks.

In [4]:
core = core.sort_values(["clientId", "submissionDate"]).copy()

# Ratios and utilization
core["repayment_ratio"] = core["actualPaymentAmount"] / (core["scheduledPaymentAmount"].replace(0, np.nan))
core["principal_completion_ratio"] = core["principalPaid"] / (core["disbursedAmount"].replace(0, np.nan))
core["utilization_ratio"] = core["disbursedAmount"] / (core["approvedAmount"].replace(0, np.nan))
core["past_due_ratio"] = core["amountPastDue"] / (core["scheduledPaymentAmount"].replace(0, np.nan))

# Arrears trends and rolling behavior per client
for col in ["daysInArrears", "amountPastDue", "repayment_ratio"]:
    core[f"{col}_rolling_mean_3"] = core.groupby("clientId")[col].transform(lambda s: s.rolling(3, min_periods=1).mean())
    core[f"{col}_rolling_std_3"] = core.groupby("clientId")[col].transform(lambda s: s.rolling(3, min_periods=1).std())

core["arrears_trend_delta"] = core.groupby("clientId")["daysInArrears"].transform(lambda s: s.diff().fillna(0))

# Volatility indicators
core["payment_volatility_3"] = core.groupby("clientId")["actualPaymentAmount"].transform(lambda s: s.rolling(3, min_periods=1).std())

# Sector benchmark (median arrears by sector)
sector_benchmark = core.groupby("industrySectorOfActivity")["daysInArrears"].median().rename("sector_daysInArrears_median")
core = core.merge(sector_benchmark, on="industrySectorOfActivity", how="left")
core["relative_arrears_vs_sector"] = core["daysInArrears"] - core["sector_daysInArrears_median"]

# Keep last known loan state per client for supervised merge with impact outcomes
core_latest = core.sort_values(["clientId", "submissionDate"]).groupby("clientId", as_index=False).tail(1)

# Merge with impact data (client-aligned, one row per survey record)
model_df = impact.merge(core_latest, left_on="unique_id", right_on="clientId", how="left", suffixes=("_impact", "_core"))

# Post-merge cleanup + additional business features
model_df["revenue_to_expense_ratio"] = model_df["revenue"] / (model_df["hh_expense"].replace(0, np.nan))
model_df["nps_net"] = model_df["nps_promoter"].fillna(0) - model_df["nps_detractor"].fillna(0)
model_df["jobs_net_3m"] = model_df["jobs_created_3m"].fillna(0) - model_df["jobs_lost_3m"].fillna(0)

for c in model_df.columns:
    if model_df[c].dtype.kind in "fiu":
        model_df[c] = model_df[c].replace([np.inf, -np.inf], np.nan)

print("Model dataset shape:", model_df.shape)
display(model_df[["unique_id", "survey_date", "risk_tier_3m", "risk_score_3m", "jobs_created_3m", "jobs_lost_3m", "revenue_3m"]].head(3))

Model dataset shape: (10000, 113)


,unique_id,survey_date,risk_tier_3m,risk_score_3m,jobs_created_3m,jobs_lost_3m,revenue_3m
0,U02254,2018-05-06,HIGH,0.761924,1,0,278.02
1,U02941,2024-06-25,MEDIUM,0.510883,0,0,372.06
2,U00585,2022-11-08,LOW,0.292304,2,0,876.59


## 5) Train/Test Split
Time-based split to avoid leakage.

In [5]:
TARGET_ORDINAL = "risk_tier_3m"
TARGET_PROB = "risk_score_3m"
TARGETS_IMPACT = ["jobs_created_3m", "jobs_lost_3m", "revenue_3m"]

# Keep rows with required targets and time field
model_df = model_df.dropna(subset=["survey_date", TARGET_ORDINAL, TARGET_PROB] + TARGETS_IMPACT).copy()
model_df = model_df.sort_values("survey_date").reset_index(drop=True)

# Ordered risk mapping (LOW < MEDIUM < HIGH)
risk_order = ["LOW", "MEDIUM", "HIGH"]
model_df[TARGET_ORDINAL] = model_df[TARGET_ORDINAL].str.upper().replace({"MID": "MEDIUM"})
model_df = model_df[model_df[TARGET_ORDINAL].isin(risk_order)].copy()

# Time split: final 20% by date as test
split_idx = int(len(model_df) * 0.8)
train_df = model_df.iloc[:split_idx].copy()
test_df = model_df.iloc[split_idx:].copy()

# Features (drop identifiers/targets/leakage-prone fields)
leakage_cols = {
    TARGET_ORDINAL, TARGET_PROB, *TARGETS_IMPACT,
    "survey_date", "survey_id", "loanNumber", "clientId", "unique_id"
}
feature_cols = [c for c in model_df.columns if c not in leakage_cols]

X_train = train_df[feature_cols]
X_test = test_df[feature_cols]

y_train_ord = train_df[TARGET_ORDINAL]
y_test_ord = test_df[TARGET_ORDINAL]

y_train_prob = train_df[TARGET_PROB]
y_test_prob = test_df[TARGET_PROB]

y_train_impact = train_df[TARGETS_IMPACT]
y_test_impact = test_df[TARGETS_IMPACT]

num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_cols),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent"))]), cat_cols),
    ],
    remainder="drop",
)

print(f"Train rows: {len(train_df)}, Test rows: {len(test_df)}")
print(f"Features: {len(feature_cols)} | Numeric: {len(num_cols)} | Categorical: {len(cat_cols)}")

Train rows: 8000, Test rows: 2000
Features: 103 | Numeric: 60 | Categorical: 43


## 6) Baseline Models
Ordinal logistic baseline for risk tier and linear/Poisson baselines for impact.

In [6]:
# Rebuild preprocess with categorical encoding (fixes: could not convert string to float: 'Male')
preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
        ]), cat_cols),
    ],
    remainder="drop",
)

# Encode ordered classes for ordinal setup
ord_map = {k: i for i, k in enumerate(risk_order)}
y_train_ord_num = y_train_ord.map(ord_map)
y_test_ord_num = y_test_ord.map(ord_map)

# Ordinal logistic approximation via multinomial logistic on ordered labels
baseline_ord = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=2000, random_state=SEED, multi_class="multinomial")),
])
baseline_ord.fit(X_train, y_train_ord_num)

ord_pred_proba = baseline_ord.predict_proba(X_test)
ord_pred_class = baseline_ord.predict(X_test)

# Probability baseline for risk_score_3m
baseline_prob = Pipeline([
    ("prep", preprocess),
    ("reg", LinearRegression()),
])
baseline_prob.fit(X_train, y_train_prob)
prob_pred = np.clip(baseline_prob.predict(X_test), 0, 1)

# Impact baselines
impact_baselines = {}
impact_preds = {}
for tgt in TARGETS_IMPACT:
    if "jobs_" in tgt:
        model = Pipeline([
            ("prep", preprocess),
            ("reg", PoissonRegressor(alpha=0.01, max_iter=1000)),
        ])
    else:
        model = Pipeline([
            ("prep", preprocess),
            ("reg", LinearRegression()),
        ])
    model.fit(X_train, y_train_impact[tgt])
    impact_baselines[tgt] = model
    impact_preds[tgt] = np.maximum(0, model.predict(X_test))

print("Baseline models trained")

Baseline models trained


## 7) Advanced Models
LightGBM/XGBoost with randomized hyperparameter search (fallback to RandomForest if unavailable).

In [ ]:
advanced_models = {}


def xgb_runtime_params(use_gpu: bool) -> Dict[str, object]:
    if use_gpu:
        return {"tree_method": "hist", "device": "cuda"}
    return {"tree_method": "hist"}


# LightGBM GPU mode requires OpenCL; force CPU to avoid "No OpenCL device found"
LGB_DEVICE = "cpu"

# Classification model for ordinal tier
if lgb is not None:
    clf = lgb.LGBMClassifier(
        random_state=SEED,
        objective="multiclass",
        num_class=3,
        device_type=LGB_DEVICE,
    )
    clf_params = {
        "clf__n_estimators": [200, 400],
        "clf__learning_rate": [0.03, 0.05, 0.1],
        "clf__num_leaves": [15, 31, 63],
        "clf__subsample": [0.8, 1.0],
    }
elif XGBClassifier is not None:
    clf = XGBClassifier(
        random_state=SEED,
        objective="multi:softprob",
        eval_metric="mlogloss",
        num_class=3,
        **xgb_runtime_params(USE_XGB_GPU),
    )
    clf_params = {
        "clf__n_estimators": [200, 400],
        "clf__learning_rate": [0.03, 0.05, 0.1],
        "clf__max_depth": [3, 5, 7],
        "clf__subsample": [0.8, 1.0],
    }
else:
    clf = RandomForestClassifier(random_state=SEED, n_estimators=400)
    clf_params = {
        "clf__max_depth": [None, 8, 12],
        "clf__min_samples_leaf": [1, 3, 5],
    }

advanced_ord = Pipeline([("prep", preprocess), ("clf", clf)])
cv = TimeSeriesSplit(n_splits=3)
search_ord = RandomizedSearchCV(
    advanced_ord,
    clf_params,
    n_iter=6,
    scoring="f1_weighted",
    cv=cv,
    random_state=SEED,
    n_jobs=-1,
)
search_ord.fit(X_train, y_train_ord_num)


def build_regressor():
    if lgb is not None:
        return lgb.LGBMRegressor(random_state=SEED, device_type=LGB_DEVICE)
    if XGBRegressor is not None:
        return XGBRegressor(random_state=SEED, eval_metric="rmse", **xgb_runtime_params(USE_XGB_GPU))
    return RandomForestRegressor(random_state=SEED, n_estimators=400)


reg_models = {}
for tgt in [TARGET_PROB] + TARGETS_IMPACT:
    reg_pipe = Pipeline([("prep", preprocess), ("reg", build_regressor())])
    reg_pipe.fit(X_train, train_df[tgt])
    reg_models[tgt] = reg_pipe

# Populate container used by downstream cells
advanced_models["risk_tier"] = search_ord.best_estimator_
advanced_models["regressors"] = reg_models

print("Advanced models trained")
print("Best ordinal model:", advanced_models["risk_tier"]["clf"].__class__.__name__)
print(f"GPU mode used -> LightGBM: {LGB_DEVICE == 'gpu'}, XGBoost: {USE_XGB_GPU}")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.704196 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1090
[LightGBM] [Info] Number of data points in the train set: 6000, number of used features: 42
[LightGBM] [Info] Start training from score -1.918457
[LightGBM] [Info] Start training from score -0.223144
[LightGBM] [Info] Start training from score -2.934324
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.194979 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1090
[LightGBM] [Info] Number of data points in the train set: 6000, number of used features: 42
[LightGBM] [Info] Start training from score -1.918457
[LightGBM] [Info] Start training from score -0.223144
[LightGBM] [Info] Start tr

KeyError: 'risk_tier'

## 8) Feature Selection
Combine Lasso, tree importance, and SHAP importance.

In [8]:
X_train_pre = preprocess.fit_transform(X_train)

# Lasso on risk_score as continuous proxy
lasso = LassoCV(cv=5, random_state=SEED).fit(X_train_pre, y_train_prob)
lasso_mask = np.abs(lasso.coef_) > 1e-6

# Tree importance from advanced risk classifier
risk_model = advanced_models["risk_tier"]
if hasattr(risk_model["clf"], "feature_importances_"):
    tree_importance = risk_model["clf"].feature_importances_
else:
    tree_importance = np.zeros(X_train_pre.shape[1])

# SHAP importance (if available)
shap_importance = np.zeros(X_train_pre.shape[1])
if shap is not None and hasattr(risk_model["clf"], "predict"):
    try:
        explainer = shap.Explainer(risk_model["clf"])
        shap_values = explainer(X_train_pre[: min(500, len(train_df))])
        shap_importance = np.mean(np.abs(shap_values.values), axis=0)
    except Exception:
        pass

# Resolve transformed feature names
feature_names = preprocess.get_feature_names_out()
fs_df = pd.DataFrame({
    "feature": feature_names,
    "lasso_selected": np.pad(lasso_mask, (0, max(0, len(feature_names) - len(lasso_mask))), constant_values=False)[: len(feature_names)],
    "tree_importance": np.pad(tree_importance, (0, max(0, len(feature_names) - len(tree_importance))), constant_values=0)[: len(feature_names)],
    "shap_importance": np.pad(shap_importance, (0, max(0, len(feature_names) - len(shap_importance))), constant_values=0)[: len(feature_names)],
})
fs_df["combined_rank"] = fs_df[["tree_importance", "shap_importance"]].sum(axis=1)
fs_df = fs_df.sort_values("combined_rank", ascending=False)

display(fs_df.head(20))

KeyError: 'risk_tier'

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

## 9) Evaluation
Compute AUC/QWK/Brier for risk and RMSE/MAE for impact, plus backtesting visuals.

In [ ]:
# Advanced predictions
adv_ord_proba = advanced_models["risk_tier"].predict_proba(X_test)
adv_ord_pred = advanced_models["risk_tier"].predict(X_test)
adv_prob_pred = np.clip(advanced_models["regressors"][TARGET_PROB].predict(X_test), 0, 1)
adv_impact_pred = {t: np.maximum(0, advanced_models["regressors"][t].predict(X_test)) for t in TARGETS_IMPACT}

# Classification metrics
y_test_onehot = pd.get_dummies(y_test_ord_num).reindex(columns=[0, 1, 2], fill_value=0)
auc_macro = roc_auc_score(y_test_onehot, adv_ord_proba, multi_class="ovr", average="macro")
qwk = cohen_kappa_score(y_test_ord_num, adv_ord_pred, weights="quadratic")
brier = brier_score_loss((y_test_ord_num == 2).astype(int), adv_ord_proba[:, 2])

# Regression metrics
metrics_rows = []
for tgt in TARGETS_IMPACT + [TARGET_PROB]:
    y_true = test_df[tgt].values
    y_hat = adv_impact_pred[tgt] if tgt in TARGETS_IMPACT else adv_prob_pred
    metrics_rows.append({
        "target": tgt,
        "rmse": mean_squared_error(y_true, y_hat, squared=False),
        "mae": mean_absolute_error(y_true, y_hat),
    })

metrics_df = pd.DataFrame(metrics_rows)
print("AUC (macro):", round(auc_macro, 4))
print("QWK:", round(qwk, 4))
print("Brier (high-risk):", round(brier, 4))
display(metrics_df)

# Backtesting-style plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(test_df["survey_date"], y_test_ord_num.values, label="Actual tier", alpha=0.8)
axes[0].plot(test_df["survey_date"], adv_ord_pred, label="Pred tier", alpha=0.8)
axes[0].set_title("Backtest: Risk Tier")
axes[0].legend()

axes[1].plot(test_df["survey_date"], test_df["revenue_3m"].values, label="Actual revenue_3m", alpha=0.8)
axes[1].plot(test_df["survey_date"], adv_impact_pred["revenue_3m"], label="Pred revenue_3m", alpha=0.8)
axes[1].set_title("Backtest: Revenue 3M")
axes[1].legend()

plt.tight_layout()
plt.show()

## 10) Calibration
Apply Platt/Isotonic calibration and plot reliability curves.

In [ ]:
# Calibrate one-vs-rest probability for HIGH risk class
high_train = (y_train_ord_num == 2).astype(int)
high_test = (y_test_ord_num == 2).astype(int)

base_binary = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=1000, random_state=SEED)),
])

cal_sigmoid = CalibratedClassifierCV(base_binary, method="sigmoid", cv=3)
cal_isotonic = CalibratedClassifierCV(base_binary, method="isotonic", cv=3)

cal_sigmoid.fit(X_train, high_train)
cal_isotonic.fit(X_train, high_train)

p_sig = cal_sigmoid.predict_proba(X_test)[:, 1]
p_iso = cal_isotonic.predict_proba(X_test)[:, 1]

fig, ax = plt.subplots(figsize=(6, 5))
for probs, label in [(adv_ord_proba[:, 2], "Raw"), (p_sig, "Platt (sigmoid)"), (p_iso, "Isotonic")]:
    frac_pos, mean_pred = calibration_curve(high_test, probs, n_bins=10)
    ax.plot(mean_pred, frac_pos, marker="o", label=label)
ax.plot([0, 1], [0, 1], "k--", label="Perfectly calibrated")
ax.set_title("Reliability Curve - High Risk Probability")
ax.set_xlabel("Predicted probability")
ax.set_ylabel("Observed frequency")
ax.legend()
plt.show()

## 11) Explainability
SHAP and partial dependence plots for model transparency.

In [ ]:
# SHAP summary (if available)
if shap is not None:
    try:
        X_sample_pre = preprocess.transform(X_test.iloc[: min(300, len(X_test))])
        model_obj = advanced_models["risk_tier"]["clf"]
        explainer = shap.Explainer(model_obj)
        shap_values = explainer(X_sample_pre)
        shap.summary_plot(shap_values, X_sample_pre, show=False)
        plt.title("SHAP Summary - Risk Tier Model")
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print("SHAP plot skipped:", e)
else:
    print("SHAP not installed; skipping SHAP plots")

# Partial dependence (fallback using baseline ordinal model)
try:
    top_features = fs_df.head(2)["feature"].tolist()
    fig, ax = plt.subplots(1, len(top_features), figsize=(6 * max(1, len(top_features)), 4))
    if len(top_features) == 1:
        ax = [ax]
    for i, feat in enumerate(top_features):
        PartialDependenceDisplay.from_estimator(
            advanced_models["risk_tier"], X_test, [feat], ax=ax[i]
        )
    plt.tight_layout()
    plt.show()
except Exception as e:
    print("Partial dependence skipped:", e)

## 12) Inference Pipeline
Reusable prediction functions and batch scoring.

In [ ]:
import joblib


def score_batch(df_features: pd.DataFrame) -> pd.DataFrame:
    out = df_features.copy()

    tier_proba = advanced_models["risk_tier"].predict_proba(df_features)
    tier_pred_num = advanced_models["risk_tier"].predict(df_features)
    inv_ord_map = {v: k for k, v in ord_map.items()}

    out["pred_risk_tier_3m"] = pd.Series(tier_pred_num).map(inv_ord_map).values
    out["pred_risk_tier_low_p"] = tier_proba[:, 0]
    out["pred_risk_tier_medium_p"] = tier_proba[:, 1]
    out["pred_risk_tier_high_p"] = tier_proba[:, 2]

    out["pred_risk_score_3m"] = np.clip(advanced_models["regressors"][TARGET_PROB].predict(df_features), 0, 1)

    for tgt in TARGETS_IMPACT:
        out[f"pred_{tgt}"] = np.maximum(0, advanced_models["regressors"][tgt].predict(df_features))

    return out


# Example batch scoring on test split
scored_test = score_batch(X_test)
scored_test_export = test_df[["unique_id", "survey_date", TARGET_ORDINAL, TARGET_PROB] + TARGETS_IMPACT].reset_index(drop=True).join(
    scored_test[[
        "pred_risk_tier_3m", "pred_risk_tier_low_p", "pred_risk_tier_medium_p", "pred_risk_tier_high_p",
        "pred_risk_score_3m", "pred_jobs_created_3m", "pred_jobs_lost_3m", "pred_revenue_3m"
    ]].reset_index(drop=True)
)

# Persist artifacts
joblib.dump(advanced_models["risk_tier"], ARTIFACTS_DIR / "risk_tier_model.joblib")
for tgt in [TARGET_PROB] + TARGETS_IMPACT:
    joblib.dump(advanced_models["regressors"][tgt], ARTIFACTS_DIR / f"{tgt}_model.joblib")

scored_test_export.to_csv(ARTIFACTS_DIR / "predictions_test.csv", index=False)
print("Saved models and predictions to:", ARTIFACTS_DIR.resolve())
display(scored_test_export.head(5))

## 13) Reporting
Generate summary tables and dashboard-style plots for decisioning.

In [ ]:
summary_metrics = {
    "auc_macro": auc_macro,
    "qwk": qwk,
    "brier_high_risk": brier,
}

summary_df = pd.DataFrame([summary_metrics])
summary_df.to_csv(ARTIFACTS_DIR / "model_summary_metrics.csv", index=False)

portfolio_view = scored_test_export.copy()
portfolio_view["risk_gap"] = portfolio_view["pred_risk_score_3m"] - portfolio_view[TARGET_PROB]
portfolio_view["revenue_gap_3m"] = portfolio_view["pred_revenue_3m"] - portfolio_view["revenue_3m"]

segment_summary = (
    portfolio_view
    .assign(pred_tier=portfolio_view["pred_risk_tier_3m"])
    .groupby("pred_tier", as_index=False)
    .agg(
        clients=("unique_id", "count"),
        mean_pred_risk=("pred_risk_score_3m", "mean"),
        mean_actual_risk=(TARGET_PROB, "mean"),
        mean_pred_revenue_3m=("pred_revenue_3m", "mean"),
        mean_actual_revenue_3m=("revenue_3m", "mean"),
    )
    .sort_values("pred_tier")
)
segment_summary.to_csv(ARTIFACTS_DIR / "segment_summary.csv", index=False)

display(summary_df)
display(segment_summary)

# Dashboard-style visuals
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

segment_summary.plot.bar(x="pred_tier", y=["mean_pred_risk", "mean_actual_risk"], ax=axes[0], title="Risk Score by Predicted Tier")
segment_summary.plot.bar(x="pred_tier", y=["mean_pred_revenue_3m", "mean_actual_revenue_3m"], ax=axes[1], title="Revenue 3M by Predicted Tier")

axes[2].hist(portfolio_view["risk_gap"], bins=20, alpha=0.8)
axes[2].set_title("Distribution of Risk Score Error")
axes[2].set_xlabel("Predicted - Actual")

plt.tight_layout()
plt.show()

print("Reporting assets exported:")
for p in ["model_summary_metrics.csv", "segment_summary.csv", "predictions_test.csv", "risk_tier_model.joblib"]:
    print(" -", ARTIFACTS_DIR / p)